# EXP-16: Enhanced Two-Level Stacking

**Competition:** SPR 2026 — Mammography Report Classification  
**Base:** EXP-14 (Two-Level Stacking) + Clinical Geometry insights  
**Enhancements over EXP-14:**
- Level 1: 9 modelos (vs 7) — adicionou SVC com C=0.5 e LGB no full text + dense
- Threshold search para TODAS as 7 classes (0-6) — organizer avisou sobre classes raras
- Mais diversidade de hiperparâmetros nos base models
- Auto-seleciona melhor blend: meta-learner vs V12-weights vs híbrido
**Runtime:** CPU only

In [ ]:
import os, re, hashlib, warnings, time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
warnings.filterwarnings('ignore')

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

N_FOLDS = 5
N_CLASSES = 7

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Class distribution:\n{train["target"].value_counts().sort_index()}')

In [ ]:
def clean_achados(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    match = re.search(r'achados:(.*?)(análise comparativa:|$)', s, flags=re.DOTALL)
    if match: s = match.group(1).strip()
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def clean_full(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def stable_hash(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def extract_dense_features(df):
    features = pd.DataFrame(index=df.index)
    text_col = df['report'].fillna('').astype(str).str.lower()
    features['report_length']   = text_col.apply(len)
    features['has_measurement'] = text_col.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    features['has_spiculation'] = text_col.str.contains(r'espiculad', regex=True).astype(int)
    features['has_distortion']  = text_col.str.contains(r'distorção arquitetural', regex=True).astype(int)
    features['has_biopsy']      = text_col.str.contains(r'biopsy|biópsia|resultado de cine|carcinoma', regex=True).astype(int)
    return csr_matrix(features.values)

train["achados"] = train["report"].apply(clean_achados)
train["full"]    = train["report"].apply(clean_full)
test["achados"]  = test["report"].apply(clean_achados)
test["full"]     = test["report"].apply(clean_full)
train["group"]   = train["report"].apply(stable_hash)

X_train_dense = extract_dense_features(train)
X_test_dense  = extract_dense_features(test)
y = train["target"].astype(int).values
groups = train["group"].values

print("Preprocessing done.")

In [ ]:
print("Building TF-IDF features...")

tfidf_A = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_A = tfidf_A.fit_transform(train["achados"].values)
X_test_A  = tfidf_A.transform(test["achados"].values)

tfidf_F = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_F = tfidf_F.fit_transform(train["full"].values)
X_test_F  = tfidf_F.transform(test["full"].values)

tfidf_F2 = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=100000))
])
X_train_F2 = tfidf_F2.fit_transform(train["full"].values)
X_test_F2  = tfidf_F2.transform(test["full"].values)

X_train_lgb = hstack([X_train_A, X_train_dense]).tocsr()
X_test_lgb  = hstack([X_test_A,  X_test_dense]).tocsr()

X_train_full_dense = hstack([X_train_F, X_train_dense]).tocsr()
X_test_full_dense  = hstack([X_test_F,  X_test_dense]).tocsr()

print(f"SVC-A: {X_train_A.shape[1]}, SVC-F: {X_train_F.shape[1]}, SVC-F2: {X_train_F2.shape[1]}")
print(f"LGB: {X_train_lgb.shape[1]}, Full+Dense: {X_train_full_dense.shape[1]}")

In [ ]:
# =============================================================================
# Level 1: GroupKFold OOF training — 9 diverse base models
# Mais diversidade: SVCs com C diferentes + LGBs com params diferentes
# =============================================================================

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(X_train_A, y, groups))

L1_MODELS = [
    # 3 SVCs no V12 features (C=1.0)
    ('svc_A', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000, C=1.0), cv=3, method='sigmoid'),
     X_train_A, X_test_A),
    ('svc_F', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000, C=1.0), cv=3, method='sigmoid'),
     X_train_F, X_test_F),
    ('svc_F2', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000, C=1.0), cv=3, method='sigmoid'),
     X_train_F2, X_test_F2),
    # SVC extra com C=0.5 (mais regularizado) — diversidade
    ('svc_F_C05', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=123, max_iter=10000, C=0.5), cv=3, method='sigmoid'),
     X_train_F, X_test_F),
    # 2 LGBs com configs diferentes
    ('lgb1', lambda: lgb.LGBMClassifier(
        class_weight='balanced', n_estimators=300, learning_rate=0.05,
        max_depth=6, random_state=42, n_jobs=-1, verbose=-1),
     X_train_lgb, X_test_lgb),
    ('lgb2', lambda: lgb.LGBMClassifier(
        class_weight='balanced', n_estimators=500, learning_rate=0.03,
        max_depth=7, num_leaves=63, min_child_samples=20,
        subsample=0.8, colsample_bytree=0.6,
        random_state=123, n_jobs=-1, verbose=-1),
     X_train_lgb, X_test_lgb),
    # LGB extra no full text + dense
    ('lgb3_full', lambda: lgb.LGBMClassifier(
        class_weight='balanced', n_estimators=400, learning_rate=0.04,
        max_depth=6, min_child_samples=30,
        random_state=456, n_jobs=-1, verbose=-1),
     X_train_full_dense, X_test_full_dense),
    # LogisticRegression
    ('lr', lambda: LogisticRegression(
        class_weight='balanced', C=1.0, max_iter=5000,
        solver='lbfgs', multi_class='multinomial', random_state=42, n_jobs=-1),
     X_train_full_dense, X_test_full_dense),
    # RidgeClassifier
    ('ridge', lambda: RidgeClassifier(class_weight='balanced', alpha=1.0),
     X_train_F, X_test_F),
]

oof_probas = {}
test_probas = {}

t0 = time.time()
for name, model_fn, X_tr, X_te in L1_MODELS:
    print(f"\n--- Level 1: {name} ---")
    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)

    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = model_fn()
        m.fit(X_tr[tr_idx], y[tr_idx])

        if hasattr(m, 'predict_proba'):
            oof[va_idx] = m.predict_proba(X_tr[va_idx])
            te_acc += m.predict_proba(X_te) / N_FOLDS
        else:
            oof[va_idx] = softmax(m.decision_function(X_tr[va_idx]))
            te_acc += softmax(m.decision_function(X_te)) / N_FOLDS

        fold_f1 = f1_score(y[va_idx], np.argmax(oof[va_idx], axis=1), average='macro')
        print(f"  Fold {fi}: F1={fold_f1:.4f}")

    oof_f1 = f1_score(y, np.argmax(oof, axis=1), average='macro')
    print(f"  >> OOF F1: {oof_f1:.4f}")
    oof_probas[name] = oof
    test_probas[name] = te_acc

elapsed = time.time() - t0
print(f"\nLevel 1 done: {len(L1_MODELS)} models in {elapsed:.0f}s")

In [ ]:
# =============================================================================
# Level 2: Meta-Learner — LogisticRegression on stacked OOF probabilities
# =============================================================================

model_names = list(oof_probas.keys())
X_meta_train = np.hstack([oof_probas[name] for name in model_names])
X_meta_test  = np.hstack([test_probas[name] for name in model_names])

print(f"Meta-features shape: {X_meta_train.shape} (= {len(model_names)} models x {N_CLASSES} classes)")

meta_oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
meta_test_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)

for fi, (tr_idx, va_idx) in enumerate(folds):
    meta_lr = LogisticRegression(
        class_weight='balanced', C=1.0, max_iter=5000,
        solver='lbfgs', multi_class='multinomial', random_state=42
    )
    meta_lr.fit(X_meta_train[tr_idx], y[tr_idx])
    meta_oof[va_idx] = meta_lr.predict_proba(X_meta_train[va_idx])
    meta_test_acc += meta_lr.predict_proba(X_meta_test) / N_FOLDS

    fold_f1 = f1_score(y[va_idx], np.argmax(meta_oof[va_idx], axis=1), average='macro')
    print(f"Meta Fold {fi}: F1={fold_f1:.4f}")

meta_oof_f1 = f1_score(y, np.argmax(meta_oof, axis=1), average='macro')
print(f"\nMeta-learner OOF macro-F1: {meta_oof_f1:.4f}")

# V12-style weighted average for comparison
oof_svc_ens = 0.25 * oof_probas['svc_A'] + 0.40 * oof_probas['svc_F'] + 0.35 * oof_probas['svc_F2']
oof_v12 = 0.70 * oof_svc_ens + 0.30 * oof_probas['lgb1']
v12_f1 = f1_score(y, np.argmax(oof_v12, axis=1), average='macro')
print(f"V12-style weighted avg OOF: {v12_f1:.4f}")

# Hybrid blend
oof_hybrid = 0.5 * meta_oof + 0.5 * oof_v12
hybrid_f1 = f1_score(y, np.argmax(oof_hybrid, axis=1), average='macro')
print(f"Hybrid (50% meta + 50% V12) OOF: {hybrid_f1:.4f}")

# Pick the best
scores = {'meta': meta_oof_f1, 'v12': v12_f1, 'hybrid': hybrid_f1}
best_name = max(scores, key=scores.get)
print(f"\n>> Best approach: {best_name} (F1={scores[best_name]:.4f})")

if best_name == 'meta':
    oof_blend = meta_oof
    te_blend = meta_test_acc
elif best_name == 'v12':
    te_svc_ens = 0.25 * test_probas['svc_A'] + 0.40 * test_probas['svc_F'] + 0.35 * test_probas['svc_F2']
    oof_blend = oof_v12
    te_blend = 0.70 * te_svc_ens + 0.30 * test_probas['lgb1']
else:
    te_svc_ens = 0.25 * test_probas['svc_A'] + 0.40 * test_probas['svc_F'] + 0.35 * test_probas['svc_F2']
    te_v12 = 0.70 * te_svc_ens + 0.30 * test_probas['lgb1']
    oof_blend = oof_hybrid
    te_blend = 0.5 * meta_test_acc + 0.5 * te_v12

baseline_preds = np.argmax(oof_blend, axis=1)
baseline_f1 = f1_score(y, baseline_preds, average='macro')
print(f"\nSelected blend OOF macro-F1: {baseline_f1:.4f}")
print(classification_report(y, baseline_preds, digits=4))

In [ ]:
# =============================================================================
# OOF-Optimized Threshold Search — TODAS as classes (0-6)
# O organizador avisou: classes raras impactam muito o macro F1
# =============================================================================

threshold_classes = [6, 5, 4, 3, 1, 0]
threshold_range = np.arange(0.05, 0.55, 0.01)

def apply_thresholds(proba, thresholds):
    """Aplica thresholds com prioridade hierárquica."""
    preds = np.argmax(proba, axis=1).copy()
    for cls in threshold_classes:
        if cls in thresholds:
            mask = proba[:, cls] > thresholds[cls]
            for higher_cls in threshold_classes:
                if higher_cls == cls:
                    break
                if higher_cls in thresholds:
                    mask = mask & (preds != higher_cls)
            preds[mask] = cls
    return preds

best_thresholds = {}
current_f1 = baseline_f1

for cls in threshold_classes:
    best_cls_f1 = current_f1
    best_cls_thr = None
    for thr in threshold_range:
        trial = {**best_thresholds, cls: thr}
        trial_preds = apply_thresholds(oof_blend, trial)
        trial_f1 = f1_score(y, trial_preds, average='macro')
        if trial_f1 > best_cls_f1:
            best_cls_f1 = trial_f1
            best_cls_thr = thr
    if best_cls_thr is not None:
        best_thresholds[cls] = best_cls_thr
        current_f1 = best_cls_f1
        print(f"Class {cls}: threshold={best_cls_thr:.2f}, macro-F1={best_cls_f1:.4f}")
    else:
        print(f"Class {cls}: no improvement from thresholds")

final_oof_preds = apply_thresholds(oof_blend, best_thresholds)
final_oof_f1 = f1_score(y, final_oof_preds, average='macro')
print(f"\nFinal OOF macro-F1 with thresholds: {final_oof_f1:.4f}")
print(f"Optimal thresholds: {best_thresholds}")
print(f"\nPer-class F1 with thresholds:")
print(classification_report(y, final_oof_preds, digits=4))

In [ ]:
# =============================================================================
# Apply thresholds + clinical guardrails → submission
# =============================================================================

test_preds = apply_thresholds(te_blend, best_thresholds)

def apply_safe_rules(row):
    text = str(row['report']).lower()
    current_pred = int(row['target'])
    if re.search(r'resultado de cine grau 3|carcinoma|\bcdis\b', text):
        return 6
    if 'espiculad' in text and 'distorção' in text and current_pred < 4:
        return 5
    return current_pred

test['target'] = test_preds
test['target'] = test.apply(apply_safe_rules, axis=1)

submission = test[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

print('Prediction distribution:')
print(submission['target'].value_counts().sort_index())
print(f'\nSubmission saved! Shape: {submission.shape}')
print(submission)